# LTL-Net+TPD 稳定性实验（账号 changyasong）

今晚运行配对种子 `42`、`3407`。明天需要第三次重复时，把 `SEEDS` 改为 `[2026]`。

运行前：开启 GPU，并 Add Input `changyasong/v5data`。

In [ ]:
SEEDS = [42, 3407]
DATA_ROOT = '/kaggle/input/datasets/changyasong/v5data/datasetv5_random811'
GIT_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
GIT_REF = 'test-new-module'
EPOCHS = 80

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
!pip install rasterio segmentation-models-pytorch -q

In [ ]:
!rm -rf /kaggle/working/lunar-linear
!git clone $GIT_URL /kaggle/working/lunar-linear
!cd /kaggle/working/lunar-linear && git checkout $GIT_REF
!cd /kaggle/working/lunar-linear && git rev-parse HEAD

In [ ]:
import os
assert os.path.isdir(DATA_ROOT), f'数据集不存在: {DATA_ROOT}'
for split, expected in [('train', 1718), ('val', 214), ('test', 216)]:
    image_dir = os.path.join(DATA_ROOT, split, 'image')
    count = len([f for f in os.listdir(image_dir) if f.lower().endswith(('.tif', '.tiff'))])
    print(split, count)
    assert count == expected, f'{split} 数量异常: {count} != {expected}'

In [ ]:
import os, sys, subprocess
repo = '/kaggle/working/lunar-linear/LTL-Net'
for seed in SEEDS:
    run_name = f'LTLNet_resnet50_TPD_seed{seed}'
    result_dir = f'/kaggle/working/result_{run_name}'
    os.makedirs(result_dir, exist_ok=True)
    log_path = os.path.join(result_dir, 'train_log.txt')
    cmd = [sys.executable, 'scripts/train_ltl.py',
           '--encoder', 'resnet50', '--detail-channels', '16',
           '--data-dir', DATA_ROOT, '--seed', str(seed),
           '--epochs', str(EPOCHS), '--run-name', run_name]
    print('\nRUN:', ' '.join(cmd), flush=True)
    with open(log_path, 'w', encoding='utf-8') as log_file:
        process = subprocess.Popen(cmd, cwd=repo, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{run_name} 失败，return code={return_code}')

In [ ]:
import json, glob
for path in sorted(glob.glob('/kaggle/working/result_LTLNet_resnet50_TPD_seed*/metrics.json')):
    with open(path, encoding='utf-8') as f:
        result = json.load(f)
    test = result['test']
    print(result['seed'], 'mIoU_all=', round(test['miou'], 4),
          'mIoU_fg=', round(test['miou_fg'], 4),
          'IoU=', [round(x, 4) for x in test['iou_per_class']])

In [ ]:
import zipfile, os, glob
for result_dir in sorted(glob.glob('/kaggle/working/result_LTLNet_resnet50_TPD_seed*')):
    zip_path = result_dir + '.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(result_dir):
            for name in files:
                full_path = os.path.join(root, name)
                zf.write(full_path, os.path.relpath(full_path, '/kaggle/working'))
    print(zip_path, f'{os.path.getsize(zip_path)/1024/1024:.1f} MB')